In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/DeepfakeAudio"

os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/models", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/outputs", exist_ok=True)

print("Project folders ready.")

Project folders ready.


In [4]:
!pip install -q kaggle librosa soundfile scikit-learn tqdm timm

In [5]:
import os

os.environ["KAGGLE_API_TOKEN"] = "KGAT_95aeb0c5e0e4e287e090a1c1a86a4fcb"

In [6]:
!kaggle datasets download -d mohammedabdeldayem/the-fake-or-real-dataset

Dataset URL: https://www.kaggle.com/datasets/mohammedabdeldayem/the-fake-or-real-dataset
License(s): GNU Lesser General Public License 3.0
100% 16.0G/16.0G [15:47<00:00, 18.2MB/s]



In [7]:
!ls -lh

total 17G
drwx------ 5 root root 4.0K Jun 13 12:44 drive
drwxr-xr-x 1 root root 4.0K Jun  4 13:39 sample_data
-rw-r--r-- 1 root root  17G Apr 16  2024 the-fake-or-real-dataset.zip


In [8]:
import zipfile

with zipfile.ZipFile(
    "the-fake-or-real-dataset.zip",
    "r"
) as z:

    print("Total files:", len(z.namelist()))

    for name in z.namelist():
        if name.startswith("for-norm"):
            print(name)
            break

Total files: 169754
for-norm/for-norm/testing/fake/file1.wav_16k.wav_norm.wav_mono.wav_silence.wav


In [9]:
import zipfile
from tqdm import tqdm

zip_path = "the-fake-or-real-dataset.zip"

with zipfile.ZipFile(zip_path, "r") as z:

    norm_files = [
        f for f in z.namelist()
        if f.startswith("for-norm/")
    ]

    print("Files to extract:", len(norm_files))

    for file in tqdm(norm_files):
        z.extract(file, "/content/dataset")

Files to extract: 69300


100%|██████████| 69300/69300 [01:30<00:00, 767.25it/s]


In [10]:
!ls /content/dataset/for-norm/for-norm

testing  training  validation


In [11]:
import os
import random
import warnings

import numpy as np
import pandas as pd

import librosa
import librosa.display

import matplotlib.pyplot as plt

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from PIL import Image

import torch.nn.functional as F

warnings.filterwarnings("ignore")

In [12]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Seed fixed:", SEED)

Seed fixed: 42


In [13]:
DATA_ROOT = "/content/dataset/for-norm/for-norm"

TRAIN_DIR = os.path.join(DATA_ROOT, "training")
VAL_DIR = os.path.join(DATA_ROOT, "validation")
TEST_DIR = os.path.join(DATA_ROOT, "testing")

print(TRAIN_DIR)
print(VAL_DIR)
print(TEST_DIR)

/content/dataset/for-norm/for-norm/training
/content/dataset/for-norm/for-norm/validation
/content/dataset/for-norm/for-norm/testing


In [14]:
def get_file_paths(split_dir):

    files = []

    fake_dir = os.path.join(split_dir, "fake")
    real_dir = os.path.join(split_dir, "real")

    for f in os.listdir(fake_dir):
        files.append(
            (os.path.join(fake_dir, f), 0)
        )

    for f in os.listdir(real_dir):
        files.append(
            (os.path.join(real_dir, f), 1)
        )

    return files

train_files = get_file_paths(TRAIN_DIR)
val_files = get_file_paths(VAL_DIR)
test_files = get_file_paths(TEST_DIR)

print("Train:", len(train_files))
print("Validation:", len(val_files))
print("Test:", len(test_files))

Train: 53868
Validation: 10798
Test: 4634


In [15]:
random.shuffle(train_files)
random.shuffle(val_files)
random.shuffle(test_files)

In [44]:
train_subset = train_files[:10000]
val_subset = val_files[:2000]
test_subset = test_files[:2000]

print(len(train_subset))
print(len(val_subset))
print(len(test_subset))

10000
2000
2000


In [45]:
class DeepfakeAudioDataset(Dataset):

    def __init__(self, file_list):
        self.file_list = file_list

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):

        audio_path, label = self.file_list[idx]

        y, sr = librosa.load(
            audio_path,
            sr=16000,
            mono=True
        )

        mel = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_mels=128
        )

        mel_db = librosa.power_to_db(
            mel,
            ref=np.max
        )

        mel_db = (
            mel_db - mel_db.min()
        ) / (
            mel_db.max() - mel_db.min() + 1e-8
        )

        mel_tensor = torch.tensor(
            mel_db,
            dtype=torch.float32
        )

        mel_tensor = mel_tensor.unsqueeze(0)

        mel_tensor = F.interpolate(
            mel_tensor.unsqueeze(0),
            size=(224,224),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        mel_tensor = mel_tensor.repeat(3,1,1)

        return mel_tensor, label

In [46]:
train_dataset = DeepfakeAudioDataset(train_subset)
val_dataset = DeepfakeAudioDataset(val_subset)
test_dataset = DeepfakeAudioDataset(test_subset)

print("Datasets created")

Datasets created


In [47]:
x, y = train_dataset[0]

print(x.shape)
print(y)

torch.Size([3, 224, 224])
0


In [48]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [49]:
import time

start = time.time()

batch_x, batch_y = next(iter(train_loader))

end = time.time()

print("Batch load time:", end - start)

print(batch_x.shape)
print(batch_y.shape)

Batch load time: 3.869969129562378
torch.Size([32, 3, 224, 224])
torch.Size([32])


In [52]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

cuda
Tesla T4


In [50]:
weights = EfficientNet_B0_Weights.DEFAULT

model = efficientnet_b0(weights=weights)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    2
)

model = model.to(device)

In [51]:
for param in model.features.parameters():
    param.requires_grad = False

for param in model.features[-2:].parameters():
    param.requires_grad = True

In [53]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=1e-4,
    weight_decay=1e-4
)

In [54]:
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    2
)

model = model.to(device)

print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=2, bias=True)
)


In [55]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

10000
2000
2000


In [57]:
def validate(model, loader):

    model.eval()

    preds = []
    labels = []

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)

            outputs = model(x)

            pred = outputs.argmax(dim=1)

            preds.extend(pred.cpu().numpy())
            labels.extend(y.numpy())

    return accuracy_score(labels, preds)

In [58]:
def train_one_epoch(model, loader):

    model.train()

    running_loss = 0

    for batch_idx, (x, y) in enumerate(loader):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(x)

        loss = criterion(outputs, y)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        if batch_idx % 50 == 0:
            print(
                f"Batch {batch_idx}/{len(loader)} "
                f"Loss: {loss.item():.4f}"
            )

    return running_loss / len(loader)

In [59]:
BEST_MODEL_PATH = "/content/drive/MyDrive/DeepfakeAudio/models/best_model_v2.pth"

NUM_EPOCHS = 5

best_val_acc = 0

for epoch in range(NUM_EPOCHS):

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

    train_loss = train_one_epoch(
        model,
        train_loader
    )

    val_acc = validate(
        model,
        val_loader
    )

    print("Train Loss:", train_loss)
    print("Validation Accuracy:", val_acc)

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print("Best model saved.")


Epoch 1/5
Batch 0/313 Loss: 0.6794
Batch 50/313 Loss: 0.4081
Batch 100/313 Loss: 0.2817
Batch 150/313 Loss: 0.2147
Batch 200/313 Loss: 0.2007
Batch 250/313 Loss: 0.1798
Batch 300/313 Loss: 0.1376
Train Loss: 0.2658514727514011
Validation Accuracy: 0.9595
Best model saved.

Epoch 2/5
Batch 0/313 Loss: 0.1609
Batch 50/313 Loss: 0.1167
Batch 100/313 Loss: 0.1646
Batch 150/313 Loss: 0.0651
Batch 200/313 Loss: 0.0306
Batch 250/313 Loss: 0.0516
Batch 300/313 Loss: 0.0191
Train Loss: 0.11721973512143183
Validation Accuracy: 0.9765
Best model saved.

Epoch 3/5
Batch 0/313 Loss: 0.1924
Batch 50/313 Loss: 0.0443
Batch 100/313 Loss: 0.0578
Batch 150/313 Loss: 0.0531
Batch 200/313 Loss: 0.0639
Batch 250/313 Loss: 0.0462
Batch 300/313 Loss: 0.0801
Train Loss: 0.07692499504016992
Validation Accuracy: 0.983
Best model saved.

Epoch 4/5
Batch 0/313 Loss: 0.0528
Batch 50/313 Loss: 0.0211
Batch 100/313 Loss: 0.0798
Batch 150/313 Loss: 0.0463
Batch 200/313 Loss: 0.0103
Batch 250/313 Loss: 0.0206
Batch 3

In [60]:
model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=device
    )
)

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for x, y in test_loader:

        x = x.to(device)

        outputs = model(x)

        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())

acc = accuracy_score(
    all_labels,
    all_preds
)

prec = precision_score(
    all_labels,
    all_preds
)

rec = recall_score(
    all_labels,
    all_preds
)

f1 = f1_score(
    all_labels,
    all_preds
)

cm = confusion_matrix(
    all_labels,
    all_preds
)

print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1 Score :", f1)

print("\nConfusion Matrix")
print(cm)

Accuracy : 0.8495
Precision: 0.777504105090312
Recall   : 0.9692937563971341
F1 Score : 0.8628701594533029

Confusion Matrix
[[752 271]
 [ 30 947]]


In [61]:
from sklearn.metrics import classification_report

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=["Fake","Real"]
    )
)

              precision    recall  f1-score   support

        Fake       0.96      0.74      0.83      1023
        Real       0.78      0.97      0.86       977

    accuracy                           0.85      2000
   macro avg       0.87      0.85      0.85      2000
weighted avg       0.87      0.85      0.85      2000



In [62]:
all_probs = []

model.eval()

with torch.no_grad():

    for x, y in test_loader:

        x = x.to(device)

        outputs = model(x)

        probs = torch.softmax(
            outputs,
            dim=1
        )[:,1]

        all_probs.extend(
            probs.cpu().numpy()
        )

In [63]:
from sklearn.metrics import roc_curve
import numpy as np

fpr, tpr, thresholds = roc_curve(
    all_labels,
    all_probs
)

fnr = 1 - tpr

eer_idx = np.nanargmin(
    np.absolute(fnr - fpr)
)

eer = (fpr[eer_idx] + fnr[eer_idx]) / 2

print("EER:", eer)

EER: 0.09851761581876815


In [64]:
BEST_MODEL_PATH = "/content/drive/MyDrive/DeepfakeAudio/models/best_model_v2.pth"

model.load_state_dict(
    torch.load(BEST_MODEL_PATH)
)

<All keys matched successfully>

In [65]:
for param in model.features.parameters():
    param.requires_grad = True

In [66]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

In [68]:
BEST_MODEL_PATH = "/content/drive/MyDrive/DeepfakeAudio/models/best_model_finetuned.pth"

NUM_EPOCHS = 2

best_val_acc = 0.9895
for epoch in range(NUM_EPOCHS):

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

    train_loss = train_one_epoch(
        model,
        train_loader
    )

    val_acc = validate(
        model,
        val_loader
    )

    print("Train Loss:", train_loss)
    print("Validation Accuracy:", val_acc)

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print("Best model saved.")


Epoch 1/2
Batch 0/313 Loss: 0.0140
Batch 50/313 Loss: 0.0070
Batch 100/313 Loss: 0.0165
Batch 150/313 Loss: 0.0235
Batch 200/313 Loss: 0.0623
Batch 250/313 Loss: 0.0442
Batch 300/313 Loss: 0.0089
Train Loss: 0.03070132269303853
Validation Accuracy: 0.993
Best model saved.

Epoch 2/2
Batch 0/313 Loss: 0.0744
Batch 50/313 Loss: 0.1166
Batch 100/313 Loss: 0.0030
Batch 150/313 Loss: 0.0061
Batch 200/313 Loss: 0.0531
Batch 250/313 Loss: 0.0021
Batch 300/313 Loss: 0.0062
Train Loss: 0.0229845080355825
Validation Accuracy: 0.996
Best model saved.


In [69]:
BEST_MODEL_PATH = "/content/drive/MyDrive/DeepfakeAudio/models/best_model_finetuned.pth"

model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=device
    )
)

<All keys matched successfully>

In [70]:
model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=device
    )
)

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for x, y in test_loader:

        x = x.to(device)

        outputs = model(x)

        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())

acc = accuracy_score(
    all_labels,
    all_preds
)

prec = precision_score(
    all_labels,
    all_preds
)

rec = recall_score(
    all_labels,
    all_preds
)

f1 = f1_score(
    all_labels,
    all_preds
)

cm = confusion_matrix(
    all_labels,
    all_preds
)

print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1 Score :", f1)

print("\nConfusion Matrix")
print(cm)

Accuracy : 0.7995
Precision: 0.7146050670640834
Recall   : 0.9815762538382804
F1 Score : 0.8270806382061233

Confusion Matrix
[[640 383]
 [ 18 959]]
